## **_Sample Modeling_** (for Reinforcement Learning with LSTM)

* 이 페이지는 "Drying Oven"의 가상 센서 데이터를 이용하여 PPO 모델을 교육하고 예측하는 예제를 구현합니다. 
* 이 페이지에서는 Liear 대신에 LSTM을 이용하도록 변경합니다.

In [11]:
# ----------------------------------------------------
# Global 변수 선언
# ----------------------------------------------------

csv_file_name = "drying_oven_sensor.csv"

#### Step 1 가상 센서 데이터 생성

**시나리오**
* 문 열림이나 외부 간섭으로 인해 내부 온도가 급격히 떨어질 때, 공기의 흐름(풍압)도 함께 불규칙하게 요동치는 상황을 반영했습니다. 보통 문이 열리면 외부 공기 유입으로 인해 압력이 순간적으로 변하고, 시스템은 이를 보정하기 위해 팬 RPM을 급격히 조절하게 됩니다.
* 풍압 요동(Pressure Fluctuation): drop_start 지점에서 풍압이 ±10.0 범위로 크게 튀게 설정했습니다. 이는 실제 환경에서 문이 열릴 때 발생하는 압력 차를 모사한 것입니다.
* 팬 RPM 연동: 풍압이 요동치면 시스템이 이를 보정하려고 하므로, RPM 값도 풍압에 따라 급격히 변화하도록 수식을 연결했습니다.
* 복구 패턴: 온도가 떨어지는 초기에 풍압이 가장 심하게 요동치고, 온도를 다시 올리는 복구 구간에서는 풍압이 서서히 안정화되는 흐름을 가집니다.
* 이 데이터를 사용하여 학습시킬 때, 온도 오차뿐만 아니라 풍압의 급격한 변화에 대해서도 마이너스 보상(Penalty)을 주면 더 안정적인 제어 로직을 만들 수 있습니다.

In [ ]:
import pandas as pd
import numpy as np
from enum import Enum

# 1. 설정
n_samples = 2000
target_temp = 80.0
target_pressure = 15.0
data = []

# 초기값 설정
curr_temp = 25.0
curr_hum = 60.0
curr_weight = 5000.0

# 구간 설정
warm_up_end = int(n_samples * 0.1)      # 10%: 초기 가열
drop_start = int(n_samples * 0.5)       # 50% 지점: 이상 발생
recovery_end = drop_start + 100         # 100스텝: 복구 구간
stable_end = int(n_samples * 0.9)       # 이후 90%까지 안정 (전체 약 80% 비중)

class HeatState(Enum):
    heating = 1.0
    stable = 2.0
    overheat = 3.0

for i in range(n_samples):
    # 기본 풍압 상태 (평상시 노이즈)
    press_noise = np.random.normal(0, 0.3)
    
    # 2. 구간별 제어 및 환경 변화
    if i < warm_up_end:
        h_state = HeatState.heating.value
        curr_temp = min(target_temp, curr_temp + np.random.uniform(1.5, 2.5))
        air_pressure = target_pressure + press_noise
        
    elif drop_start <= i < drop_start + 15: 
        # [급격한 하락 및 풍압 요동] - 문 열림/외부 간섭 발생
        h_state = HeatState.stable.value
        curr_temp -= np.random.uniform(4.0, 6.0) # 온도 급락
        # 풍압이 위아래로 크게 요동 (외풍 유입 모사)
        air_pressure = target_pressure + np.random.uniform(-10.0, 10.0) 
        
    elif drop_start + 15 <= i < recovery_end:
        # [복구 구간] - 다시 목표치를 향해 제어
        h_state = HeatState.heating.value
        curr_temp = min(target_temp, curr_temp + np.random.uniform(2.0, 3.5))
        # 풍압을 다시 잡기 위해 과하게 작동하는 상황
        air_pressure = target_pressure + np.random.uniform(-2.0, 4.0)
        
    elif i < stable_end:
        # [안정 구간 - 약 80% 비중]
        h_state = HeatState.stable.value
        curr_temp = target_temp + np.random.uniform(-0.5, 0.5)
        air_pressure = target_pressure + press_noise
        
    else:
        # [마지막 변동 구간]
        h_state = HeatState.overheat.value
        curr_temp += np.random.uniform(0.2, 1.0)
        air_pressure = target_pressure + np.random.uniform(-1.0, 1.0)

    # 3. 물리적 상관관계 반영
    # RPM은 풍압을 만들기 위한 제어 결과값 (P ∝ RPM^2 역산)
    curr_rpm = np.sqrt(max(0.1, air_pressure) / 5.0) * 1000 + np.random.normal(0, 15)
    
    # 나머지 센서값
    curr_surf_temp = curr_temp - np.random.uniform(1.0, 3.0)
    curr_hum = max(10.0, 60.0 - (curr_temp - 25.0) * 0.8 + np.random.normal(0, 1))
    curr_weight -= np.random.uniform(0.1, 0.4)

    data.append([
        round(curr_temp, 2), round(curr_hum, 2), round(curr_weight, 2),
        round(curr_surf_temp, 2), round(curr_rpm, 1), round(air_pressure, 3),
        h_state
    ])

# 4. 데이터프레임 생성
df = pd.DataFrame(data, columns=['internal_temp', 'humidity', 'weight', 'surface_temp', 'fan_rpm', 'air_pressure', 'heater_state'])

# 결과 확인 (요동치는 구간 출력)
print("--- 이상 발생 및 복구 구간 데이터 (500~520번) ---")
print(df.iloc[drop_start-2:drop_start+18][['internal_temp', 'air_pressure', 'fan_rpm', 'heater_state']])

df.to_csv(csv_file_name)


--- 이상 발생 및 복구 구간 데이터 (500~520번) ---
      internal_temp  air_pressure  fan_rpm  heater_state
998           79.61        14.777   1725.8           2.0
999           80.33        15.567   1780.2           2.0
1000          75.36        10.713   1456.5           2.0
1001          71.17        14.789   1721.7           2.0
1002          66.00        12.777   1580.9           2.0
1003          60.68        22.474   2117.4           2.0
1004          55.64         7.886   1237.3           2.0
1005          51.24        14.736   1735.8           2.0
1006          46.83        22.585   2137.1           2.0
1007          42.77        18.766   1940.8           2.0
1008          38.01        11.149   1490.6           2.0
1009          32.08        21.555   2057.6           2.0
1010          26.93        14.278   1716.4           2.0
1011          21.56        22.530   2132.8           2.0
1012          16.85        18.780   1932.6           2.0
1013          11.75         9.097   1376.1         

#### Step 2 Env 생성

##### 2.1 보상 함수 (풍압 중심 제어)

데이터에 급격한 온도 하락과 풍압 요동 시나리오가 포함되었으므로, 강화학습 에이전트가 이러한 비정상 상황(Abnormal State)을 빠르게 감지하고 복구하도록 유도하는 보상 함수(Reward Function)를 설계했습니다.

단순히 현재 온도만 보는 것이 아니라, 풍압의 안정성과 제어의 일관성을 모두 고려한 코드입니다.
이 코드는 Gymnasium 환경 내에서 앞서 만든 가상 데이터의 물리 법칙(온도 하락, 풍압 요동 등)을 시뮬레이션하며 에이전트를 학습시킵니다.

보상 함수에서 풍압(air_pressure)의 비중을 대폭 높이고, 에이전트가 풍압을 목표치(15Pa)에 맞추기 위해 팬 RPM을 정밀하게 제어하도록 로직을 수정했습니다.

이제 온도는 기본적인 제어 대상이 되며, 풍압의 안정성이 보상의 핵심 지표가 됩니다.

**_보상 함수는 에이전트가 다음 세 가지 목표를 동시에 달성하도록 유도합니다._**
- 온도 유지: 목표 온도(80도)와의 차이 최소화
- 풍압 안정: 목표 풍압(15Pa)과의 차이 최소화 및 요동 방지
- 제어 효율: 불필요하게 팬 RPM이나 히터를 과하게 조작하지 않음 (Penalty)

**_파이프라인의 핵심 구성 요소_**

* Gymnasium Env Interface: step() 함수 내에 에이전트의 액션이 실제 온도와 풍압에 어떻게 영향을 주는지 물리 법칙을 정의했습니다.
* PPO 알고리즘: Stable Baselines3의 PPO는 제어 시스템에서 안정적인 성능을 보여주는 대표적인 강화학습 알고리즘입니다.
* Quadratic Penalty: 온도 오차를 제곱(temp_err**2)하여 벌점을 부여함으로써, 목표 온도에서 멀어질수록 에이전트가 더 강하게 반응하도록 유도했습니다.
* Control Smoothing: ctrl_penalty를 통해 히터나 팬을 너무 급격하게 조작하지 않도록 제한하여 실제 기계의 마모를 고려했습니다.
이 파이프라인을 실행하면 에이전트가 초기 25도에서 시작하여 80도 온도와 15Pa 풍압을 찾아가고 유지하는 법을 스스로 터득하게 됩니다.

**_보상 로직의 특징_**
- Quadratic Pressure Penalty: 풍압 오차에 제곱(\^2)과 가중치(5.0)를 적용하여, 에이전트가 풍압을 15Pa에 맞추는 것을 최우선 순위로 학습하게 했습니다.
- Pressure Delta Reward: 풍압이 목표치로 접근하는 '방향성'에 보상을 주어, 급격한 요동 구간에서도 에이전트가 포기하지 않고 제어력을 유지하도록 돕습니다.
- Complex Interaction: 풍압이 높아지면 온도가 낮아지는 냉각 효과가 반영되어 있으므로, 에이전트는 온도를 지키면서도 풍압을 최적화하는 미세한 밸런스를 찾게 됩니다.

In [1]:
import gymnasium as gym
from gymnasium import spaces
import numpy as np
from stable_baselines3 import PPO
import matplotlib.pyplot as plt

class DryingOvenEnv(gym.Env):
    def __init__(self):
        super(DryingOvenEnv, self).__init__()
        # 제어: [히터 출력 증감, 팬 RPM 증감]
        self.action_space = spaces.Box(low=-1.0, high=1.0, shape=(2,), dtype=np.float32)
        # 관측: [내부온도, 습도, 무게, 표면온도, 현재RPM, 풍압]
        self.observation_space = spaces.Box(low=-np.inf, high=np.inf, shape=(6,), dtype=np.float32)
        
        self.target_temp = 80.0
        self.target_press = 15.0 # 보상의 핵심 목표
        self.reset()

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.state = np.array([25.0, 60.0, 5000.0, 24.0, 1500.0, 15.0], dtype=np.float32)
        self.prev_press_error = 0.0
        self.steps = 0
        return self.state, {}

    def step(self, action):
        curr_temp, hum, weight, surf_temp, rpm, press = self.state
        h_ctrl, r_ctrl = action
        
        # 1. 물리 시뮬레이션 (RPM이 풍압에 직접 영향)
        new_rpm = np.clip(rpm + (r_ctrl * 200), 500, 3500)
        # 풍압 물리 법칙: P = (RPM/1000)^2 * 5 + 외부 요동(noise)
        new_press = (new_rpm / 1000)**2 * 5.0 + np.random.normal(0, 0.2)
        
        # 온도는 히터와 풍압(냉각효과)의 영향을 받음
        new_temp = curr_temp + (h_ctrl * 4.0) - (new_press * 0.15)
        
        self.state = np.array([new_temp, hum-0.1, weight-0.1, new_temp*0.9, new_rpm, new_press], dtype=np.float32)
        
        # --- 핵심 수정: 풍압 중심의 보상 설계 (Reward Design) ---
        press_error = abs(self.target_press - new_press)
        temp_error = abs(self.target_temp - new_temp)
        
        # 1) 풍압 보상 (매우 강력한 페널티): 목표 풍압에서 멀어질수록 벌점 증가
        r_press = -(press_error ** 2) * 5.0 
        
        # 2) 풍압 안정성 보상: 이전 스텝보다 풍압 오차가 줄어들면 추가 보상
        r_press_delta = (self.prev_press_error - press_error) * 2.0
        
        # 3) 온도 보상: 기본 수준 유지
        r_temp = -(temp_error ** 2) * 0.5
        
        # 최종 보상 합산 (풍압 가중치 극대화)
        reward = r_press + r_press_delta + r_temp
        
        self.prev_press_error = press_error
        self.steps += 1
        
        terminated = bool(new_temp > 130 or new_press > 50)
        truncated = self.steps >= 500
        
        return self.state, reward, terminated, truncated, {}

I0000 00:00:1774150881.648379   51287 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1774150882.543467   51287 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI AVX_VNNI_INT8 AVX_NE_CONVERT FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1774150885.831030   51287 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


#### Step 3 모방 학습(Behavioral Cloning)으로 PPO 초기화하기

##### Step 3.1 가상 데이터를 Transitions 데이터로 변환

In [14]:
import pandas as pd
import torch
from torch.utils.data import DataLoader, TensorDataset

df = pd.read_csv(csv_file_name)
obs = df[['internal_temp', 'humidity', 'weight', 'surface_temp', 'fan_rpm', 'air_pressure']]
acts = df[['heater_state', 'fan_rpm']]
obs = torch.tensor(obs.values, dtype=torch.float32)
acts = torch.tensor(acts.values, dtype=torch.float32)
dataset = TensorDataset(obs, acts)
# loader = DataLoader(dataset, batch_size=64, shuffle=True)
loader = DataLoader(dataset, batch_size=32, shuffle=True)
print(f'obs\'s len = {len(obs)}')


obs's len = 2000


##### 3.2 Trasitions 데이터로 모방 학습 하기

* pytorch를 이용
* 가상 데이터 준비 및 BC 학습 (Pre-training)
* 가상 데이터에서 (State) -> (Action) 관계를 지도 학습(Supervised Learning)으로 먼저 익힙니다.

In [24]:
import gymnasium as gym
import torch as th
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sb3_contrib import RecurrentPPO


def train_bc_model_with_lstm(env_cls: gym.Env, loader,  epochs=20, device="cpu"):
    # --- 2. 모델 및 환경 설정 ---
    env = env_cls()
    bc_model = RecurrentPPO("MlpLstmPolicy", env, verbose=1)

    # 학습할 정책(Policy) 네트워크 추출
    policy = bc_model.policy
    optimizer = optim.Adam(policy.parameters(), lr=1e-3)
    loss_fn = nn.CrossEntropyLoss() # 이산 행동일 경우
    n_layers = 1
    n_envs = 1
    hidden_size = 256

    # --- 3. 사전 훈련 (BC) 루프 ---
    policy.train()
    print("LSTM 정책 사전 훈련 시작...")
    for epoch in range(epochs):
        total_loss = 0
        print(f'epochs={epoch}/{epochs}')
        for batch_obs, batch_actions in loader:
            optimizer.zero_grad()
        
            # RecurrentPPO의 정책망은 (obs, lstm_states, episode_starts)를 입력으로 받음
            # 사전 훈련 시에는 매 시퀀스 시작마다 상태를 초기화(None)한다고 가정
            # distribution은 액션의 확률 분포를 반환함
            batch_size_total = batch_obs.shape[0]
            lstm_states = (
                th.zeros(n_layers, n_envs, hidden_size).to(device),
                th.zeros(n_layers, n_envs, hidden_size).to(device)
            )
            # lstm_states = th.zeros((batch_size_total,), dtype=th.float32).to(device)
            episode_starts = th.zeros((batch_size_total,), dtype=th.float32).to(device)
            # 예: 시퀀스 길이가 10이라면 0, 10, 20... 인덱스를 1.0으로 설정
            seq_len = 10 
            episode_starts[::seq_len] = 1.0
            distribution, _ = policy.get_distribution(batch_obs, lstm_states, episode_starts)
        
            """
            action_dim은 에이전트가 한 번에 결정해야 하는 행동(Action)의 가짓수나
            차원을 의미합니다. 환경의 특성에 따라 다음과 같이 결정됩니다.
            """
            action_dim = env.action_space.shape[0]

            run_this1 = False
            if run_this1:
                # 전문가 행동과의 차이 계산 (Log Likelihood 등을 사용할 수도 있음)
                # 여기서는 간단히 CrossEntropy를 위해 logits 사용
                logits = distribution.distribution.logits
                # 차원 맞추기: (batch * seq, action_dim) vs (batch * seq)
                loss = loss_fn(logits.view(-1, action_dim), batch_actions.view(-1))
                loss.backward()
                optimizer.step()
            else:
                log_prob = distribution.log_prob(batch_actions.view(-1, action_dim))
                loss = -log_prob.mean()  # Negative Log Likelihood (NLL)
                # 4. 역전파 및 최적화
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
        
            total_loss += loss.item()
        
        print(f"Epoch {epoch+1}/{epochs}, Loss: {total_loss/len(loader):.4f}")

    # --- 4. 훈련된 가중치 저장 ---
    return bc_model

print(loader)
bc_model = train_bc_model_with_lstm(env_cls=DryingOvenEnv, loader=loader, epochs=3)


Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
LSTM 정책 사전 훈련 시작...
epochs=0/3
Epoch 1/3, Loss: 1413368.0278
epochs=1/3
Epoch 2/3, Loss: 1246949.9504
epochs=2/3
Epoch 3/3, Loss: 1108468.8234


#### 4 PPO 모델 생성

* pytorch를 이용하여 생성 된 policy 이용

* 파라미터 설명

  * train_Params = {
    * "steps": 하나의 echo에 대한 분할 횟수, exam : 2048
    * "epochs": rpeat count, exam : 10
    * "learning_rate": 3e-4,
    * "device": "cpu" }
  * learning_timesteps = 하나의 epoch에 투입되는 총 데이터, exam : 200000

In [17]:
def train_with_params(train_params, env, bc_policy, learning_timesteps, model_name, is_torch=True):
    """
     모델 설정 (PPO 알고리즘 사용)
     MlpPolicy: 센서 데이터(벡터) 처리에 적합한 다층 퍼셉트론 신경망
    
    ; param train_params : dict
    ; env : DryingOvenEnv
    ; bc_policy : BC_Policy
    ; is_torch: boolean
    ; model_name: str
    """
    ppo_model = PPO("MlpPolicy",
                    env,
                    verbose=1,
                    n_steps=train_Params["steps"],
                    n_epochs=train_Params["epochs"],
                    learning_rate=train_Params["learning_rate"],
                    device=train_Params["device"])

    # BC 모델의 가중치를 PPO의 actor 네트워크로 복사
    # SB3의 policy 구조에 맞춰 매핑 (간략화된 예시)
    if is_torch:
        with torch.no_grad():
            ppo_model.policy.action_net.weight.copy_(bc_policy.net[-1].weight)
            ppo_model.policy.action_net.bias.copy_(bc_policy.net[-1].bias)

    # 5. 강화학습으로 Fine-tuning (전이 학습)
    print("BC 정책을 기반으로 RL Fine-tuning 시작...")
    ppo_model.learn(total_timesteps=learning_timesteps) # 2만 번의 시행착오를 통해 학습

    # 6. 결과 저장
    ppo_model.save(model_name)
    print(f"모델 저장 완료: {model_name}.zip")
    return ppo_model_torch

In [21]:
# ----------------------------------------
# light train
# ----------------------------------------

env_4_light = DryingOvenEnv() # 앞서 정의한 풍압 중심 Env
train_Params = {
    "steps": 128, # 하나의 echo에 대한 분할 횟수
    "epochs": 1, # rpeat count
    "learning_rate": 3e-4,
    "device": "cpu"
}
print(f'policy = {bc_model}')
learning_timesteps = 2000 # 하나의 epoch에 투입되는 총 데이터 수
train_with_params(train_params=train_Params,
                  env=env_4_light,
                  bc_policy=bc_model,
                  learning_timesteps=learning_timesteps,
                  model_name="drying_oven_v1",
                  is_torch=True)

policy = BC_Policy(
  (net): Sequential(
    (0): Linear(in_features=6, out_features=64, bias=True)
    (1): Tanh()
    (2): Linear(in_features=64, out_features=64, bias=True)
    (3): Tanh()
    (4): Linear(in_features=64, out_features=2, bias=True)
  )
)
Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
BC 정책을 기반으로 RL Fine-tuning 시작...
----------------------------------
| rollout/           |           |
|    ep_len_mean     | 15        |
|    ep_rew_mean     | -7.41e+04 |
| time/              |           |
|    fps             | 4808      |
|    iterations      | 1         |
|    time_elapsed    | 0         |
|    total_timesteps | 128       |
----------------------------------
--------------------------------------------
| rollout/                |                |
|    ep_len_mean          | 16.1           |
|    ep_rew_mean          | -7.57e+04      |
| time/                   |                |
|    fps                  | 1455         

In [ ]:
# ----------------------------------------
# Heavy Train
# ----------------------------------------

env_4_heavy = DryingOvenEnv() # 앞서 정의한 풍압 중심 Env
train_Params = {
    "steps": 2048,
    "epochs": 10,
    "learning_rate": 3e-4,
    "device": "cpu"
}
learning_timesteps = 200000
train_with_params(train_params=train_Params,
                  env=env_4_heavy,
                  bc_policy=bc_model,
                  learning_timesteps=learning_timesteps,
                  model_name="drying_oven_v1",
                  is_torch=True)


Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
BC 정책을 기반으로 RL Fine-tuning 시작...
----------------------------------
| rollout/           |           |
|    ep_len_mean     | 9.01      |
|    ep_rew_mean     | -4.46e+04 |
| time/              |           |
|    fps             | 4687      |
|    iterations      | 1         |
|    time_elapsed    | 0         |
|    total_timesteps | 2048      |
----------------------------------
--------------------------------------------
| rollout/                |                |
|    ep_len_mean          | 9.02           |
|    ep_rew_mean          | -4.45e+04      |
| time/                   |                |
|    fps                  | 234            |
|    iterations           | 2              |
|    time_elapsed         | 17             |
|    total_timesteps      | 4096           |
| train/                  |                |
|    approx_kl            | 0.087495014    |
|    clip_fraction        | 

##### 5. Test

In [ ]:
# 테스트 실행
model = ppo_model_torch

obs, _ = env.reset()
press_history = []

for _ in range(100):
    action, _ = model.predict(obs)
    obs, r, _, _, _ = env.step(action)
    press_history.append(obs[5]) # air_pressure 기록

plt.plot(press_history, label='Air Pressure (Pa)')
plt.axhline(15, color='r', linestyle='--', label='Target')
plt.title("Reward-Driven Pressure Control")
plt.legend()
plt.show()

#### 6. 센서 데이터 읽어서 예측 결과를 모터에 전송하기

** 센서 데이터 읽기 **

* 실제 환경에 적용하기 위해 외부 센서 데이터(CSV, API, 또는 PLC 통신 등)를 실시간으로 읽어와서 학습된 BC-PPO 모델에 입력하고, 제어 명령(Action)을 출력하는 루프를 추가했습니다.
* 센서 데이터를 읽어오는 부분은 실제 환경에 맞춰 교체할 수 있도록 read_external_sensors() 함수로 추상화했습니다.
* 주요 연동 포인트
  * Observation 정규화: 학습 때 사용한 데이터의 범위와 실제 센서 데이터의 범위가 크게 다를 경우, (obs - mean) / std와 같은 Scaling 과정을 read_external_sensors 내에 추가해야 모델이 정확하게 판단합니다.
  * Deterministic Prediction: 실전 제어에서는 확률적인 탐색보다는 모델이 가장 좋다고 판단하는 확정적인 값(deterministic=True)을 사용하는 것이 안전합니다.
  * Action Mapping: 모델의 출력값은 보통 -1 ~ 1 사이의 수치입니다. 이를 실제 히터의 전압(V), 전류(mA), 혹은 PWM % 값으로 변환하는 매핑 로직이 send_control_signal에 필요합니다.

** 모터 제어 **

* 학습된 BC-PPO 모델의 예측값(Action)을 받아 실제 모터의 RPM 제어 신호로 변환하고, 이를 장비(모터 드라이버 등)에 전달하는 과정을 포함한 최종 제어 루프입니다.
* 보통 산업용 모터는 Modbus TCP나 0~10V 아날로그 신호 등을 사용하므로, 이를 모사한 변환 로직을 apply_motor_control 함수에 추가했습니다.
* 코드 핵심 포인트
  * RPM Delta 제어: 모델의 출력을 절대적 RPM 값이 아닌 변화량(rpm_delta)으로 사용했습니다. 이는 모터에 급격한 부하가 걸리는 것을 방지하고 부드러운 가속/감속을 가능하게 합니다.
  * Hard Clipping: np.clip을 사용하여 모델이 실수로 장비의 물리적 한계(500~3000 RPM)를 넘는 명령을 내려도 안전하게 차단합니다.
Global State 유지: CURRENT_MOTOR_RPM을 추적하여 현재 상태를 기준으로 다음 제어량을 결정하도록 설계했습니다.

In [8]:
import time
import numpy as np
from stable_baselines3 import PPO

# 1. 학습된 모델 로드
model = PPO.load("bc_ppo_oven_model")

# [설정] 모터 사양 및 제어 범위
MIN_RPM = 500
MAX_RPM = 3000
CURRENT_MOTOR_RPM = 1500  # 초기 가동 RPM

def read_external_sensors():
    """
    외부 센서로부터 현재 상태를 읽어옴 (예시 데이터)
    순서: [internal_temp, humidity, weight, surface_temp, fan_rpm, air_pressure]
    """
    # 실제 환경에서는 PLC/센서 API 호출 결과가 들어감
    return np.array([79.2, 30.1, 4900.5, 76.0, CURRENT_MOTOR_RPM, 14.5], dtype=np.float32)

def apply_motor_control(fan_action):
    """
    모델의 Action(-1 ~ 1)을 실제 모터 RPM 값으로 변환하여 전송
    """
    global CURRENT_MOTOR_RPM
    
    # 1. Action(-1 ~ 1)을 RPM 변화량으로 매핑 (예: 한 번에 최대 200 RPM 증감)
    rpm_delta = fan_action * 200
    
    # 2. 새로운 목표 RPM 계산 및 하드웨어 제한(Limit) 적용
    target_rpm = np.clip(CURRENT_MOTOR_RPM + rpm_delta, MIN_RPM, MAX_RPM)
    
    # 3. 실제 모터 드라이버에 명령 전달 (예: Modbus Write 또는 DAC 출력)
    # write_to_motor_driver(target_rpm) # 실제 통신 함수 가정
    
    CURRENT_MOTOR_RPM = target_rpm # 현재 상태 업데이트
    return target_rpm

def apply_heater_control(heater_action):
    """히터 제어 신호 변환 (예: 0~100% PWM)"""
    heater_power = np.clip((heater_action + 1) * 50, 0, 100)
    # write_to_heater_relay(heater_power)
    return heater_power

# 2. 실시간 제어 루프
print("--- Drying Oven AI-Driven Motor & Heater Control Start ---")

try:
    while True:
        # (1) 센서 데이터 수집
        obs = read_external_sensors()
        
        # (2) 모델 추론 (Deterministic=True로 안정성 확보)
        # obs shape: (6,) -> (1, 6)으로 모델 입력
        action, _ = model.predict(obs, deterministic=True)
        
        # action[0]: 히터 제어, action[1]: 팬(풍압) 제어
        h_action, f_action = action[0], action[1]
        
        # (3) 하드웨어 제어 명령 실행
        target_rpm = apply_motor_control(f_action)
        heater_pwr = apply_heater_control(h_action)
        
        # (4) 모니터링 출력
        print(f"[상태] 온도: {obs[0]:.1f}°C | 풍압: {obs[5]:.2f}Pa")
        print(f"[제어] 목표 RPM: {target_rpm:.0f} | 히터: {heater_pwr:.1f}%")
        print("-" * 40)
        
        # 제어 주기 (예: 2초마다 갱신)
        time.sleep(2.0)

except KeyboardInterrupt:
    print("\n--- 시스템 안전 종료 및 모터 정지 ---")
    # apply_motor_control(-1) # 모터 최소화 또는 정지 로직


FileNotFoundError: [Errno 2] No such file or directory: 'bc_ppo_oven_model.zip'